### Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

### Tokens

In [ ]:
colors = ['red', 'green', 'blue','yellow', 'purple', 'orange', 'pink', 'brown', 'gray']
shapes = ['circle', 'square', 'triangle']
positions = ['center','top','bottom','left','right','top-left','top-right','bottom-left','bottom-right']

### Constants

In [ ]:
embedding_dim = 64
num_encoder_blocks = 10


### Image Encoder
- This is a CNN based architecture . 
- Uses Relu for Activation
- **Standard CNN Input Shape:** is : [B,C, H, W] = [batch_size, num_channels, height, width]


In [ ]:
class image_encoder(nn.Module):
    def __init__(self, embed_dim=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(kernel_size = 3, stride=2, padding = 1, in_channels = 3, out_channels = 32),
            nn.ReLU(inplace=True),
            nn.Conv2d(kernel_size = 3, stride=2, padding = 1, in_channels = 32, out_channels = 64),
            nn.ReLU(inplace=True),
            nn.Conv2d(kernel_size = 3, stride=2, padding = 1, in_channels = 64, out_channels = 128),
            nn.ReLU(inplace=True),
            nn.Conv2d(kernel_size = 3, stride=2, padding = 1, in_channels = 128, out_channels = 256),
            nn.ReLU(inplace=True)
        ) # [N, C,H, W]

        '''
        # Note
        i)   adaptive average pool ensures the specified output size i.e. in this case 1x1 . 
        Hence it might be better than the ordinary average pool
        ii)  after average pool/adaptive average pool . Need to flatten output before passing through fc
        iii) x = x.mean(dim=[2,3]). This takes the average along dim 2,3 = (H,W). And it flattens things. 
        x. mean is better/preferred in modern CNN embedding layers. 
        I've deliberately used AdaptiveAvgPool2D specifically to demonstrate that between conv2ds and fully connected layers ,
        a flattening is required
        '''
        self.avg_pool = nn.AdaptiveAvgPool2d(output_size = (1,1))# [N,C,1, 1]
        self.fc      = nn.Linear(in_features = 256, out_features = embed_dim) #[N,256] to [N,64]
        self.layer_norm = nn.LayerNorm(normalized_shape= embed_dim)
        
    def forward(self,x):
        x= self.conv(x)           # [N, C,H, W]
        x= self.avg_pool(x)       # [N, C=256,1, 1]        
        x= x.flatten(start_dim=1) # [N, C=256], start_dim=1 means start flattening from dimension 1.
        x= self.fc(x)             # [N, C=64]
        x= self.layer_norm(x)

        return x
        

# Text Encoder
- This is a Transformer based architecture .
- Mutli Head Attention has no activations. Softmax is the Non Linearity
- Multi Layer Perceptron has GELU Activations
- **Standard Transformer Input Shape:** is : [B,S,E] = [batch_size, sequence_length, embedding_dimension]
  So all modules, Multi Layer Perceptrons, Multi Attention Heads all use the same shape of X

In [ ]:
class SingleHeadAttention(nn.Module):
    '''
    The standard Transformer Input Shape [ B,S,E] = [batch_size, sequence_length, embedding_dimension]
    Single Head Attention deals with inputs of this size
    Using variables attn_score, attn_weights etc. for clarity 
    '''
    def __init__(self, embed_dim):
        super().__init__()
        self.embed_dim = embed_dim
        self.scale = embed_dim ** 0.5
        self.k = nn.Linear(embed_dim, embed_dim)
        self.q = nn.Linear(embed_dim, embed_dim)
        self.v = nn.Linear(embed_dim, embed_dim)
        
    def forward(self,x):
        # Calculate K, Q, V
        K = self.k(x) # [B,S,E]
        Q = self.q(x) # [B,S,E]
        V = self.v(x) # [B,S,E]
        
        # Attention Scores and Weights
        attn_score = Q @ K.transpose(-2,-1)                      # [B,S,E] * [B,E,S] = [B,S,S]
        attn_score_scaled = attn_score/self.scale                # [B,S,S]
        attn_weights = torch.softmax(attn_score_scaled, dim= -1) # [B,S,S]

        # Attention (Weighted Attention)
        attn = attn_weights @ V                                  # [B,S,S] * [B,S,E] = [B,S,E]
        
        return attn

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_attention_heads):
        super().__init__()
        self.embed_dim = embed_dim
        self.mha_wts = []
        for _ in range(0, num_attention_heads):
            self.mha_wts.append(single_head_attention(embed_dim))
        

    def forward(self,x):
        mha = []
        for i in range(0, num_attention_heads):
            mha.append(self.mha_wts[i](x)))

        x = torch.cat(mha, dim =1) # concatenate along the number of channels
        return x


class MultiHeadAttention(nn.Module):
    '''
    Multi Head Attention: Uses Several Single Attention Heads
    Newbie mistakes to avoid
    i)  self.heads should not be a mere python list like self.heads =[]. self.heads should be a nn.ModuleList instead
    ii) the single head embedding dimension should be a factor of the full multi head attention
    
    '''
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.single_head_embed_dim = embed_dim // num_heads

        self.heads = nn.ModuleList(
            [SingleHeadAttention(self.head_dim) for _ in range(num_heads)]
        )

        self.out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        # x: [B, S, E]
        head_outputs = torch.cat(
            [head(x) for head in self.heads],
            dim=-1  # concatenate on embedding dimension
        )  # [B, S, E]

        return self.out(head_outputs

class MultiLayerPerceptron(nn.Module):
    '''
    Uses the standard 2 layer Perceptron. i.e fc1, GELU, fc2
    Modern transformers use GELU since it is smoother. 
    Hidden dimension is set to a default of 4*embedding dimension
    '''
    def __init__(self, embed_dim, hidden_dim = None):
        super().__init__()
        hidden_dim = hidden_dim or embed_dim*4
        self.fc1  = nn.Linear(embed_dim, hidden_dim)
        self.gelu = nn.GELU()
        self.fc2  = nn.Linear(hidden_dim, embed_dim)

    def forward(self,x):
        x = self.fc1(x)
        x = self.gelu(x)
        x = self.fc2(x)
        return x
            


In [ ]:
class single_encoder_block(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.mha = multi_head_attention()
        self.layer_norm1 = nn.LayerNorm(normalized_shape= embed_dim)
        self.mlp = mulit_layer_perceptron()
        self.layer_norm2 = nn.LayerNorm(normalized_shape= embed_dim)
        

    def forward(self,x):
        x_start = x
        x = self.mha(x)
        x = self.layer_norm1(x)
        x = x_start + x

        x_start = x
        x = self.mlp(x)
        x = self.layer_norm2(x)
        x = x_start + x
        return x


class encoder(nn.Module):
    def __init__(self, num_encoders):
        super().__init__()
        self.num_encoders   = num_encoders
        self.single_encoder = single_encoder_block();
        
    def forward(self,x):
        for _ in range(0,self.num_encoders):
            x = self.single_encoder(x)
        return x
        

In [1]:
import numpy as np

def standardize_vector(vector):
    """
    Standardizes a vector to have a mean of 0 and a standard deviation of 1.
    """
    # Convert to numpy array for efficient operations
    vector = np.asarray(vector, dtype=float)

    # Calculate the mean
    mean = np.mean(vector)

    # Calculate the standard deviation
    # ddof=0 is used by default in np.std, which is suitable for population statistics
    std_dev = np.std(vector)

    # Handle the case where standard deviation is zero to prevent division errors
    if std_dev == 0:
        return np.zeros_like(vector)
    
    # Apply the standardization formula
    normalized_vector = (vector - mean) / std_dev
    return normalized_vector

# Example usage:
data = [2, 3, 5, 6, 7, 4, 8, 7, 6]
normalized_data = standardize_vector(data)

print("Original Data:", data)
print("Normalized Data:", normalized_data)
print("Mean of normalized data (approx 0):", np.mean(normalized_data))
print("Std Dev of normalized data (approx 1):", np.std(normalized_data))

Original Data: [2, 3, 5, 6, 7, 4, 8, 7, 6]
Normalized Data: [-1.76776695 -1.23743687 -0.1767767   0.35355339  0.88388348 -0.70710678
  1.41421356  0.88388348  0.35355339]
Mean of normalized data (approx 0): 1.4186183092432555e-16
Std Dev of normalized data (approx 1): 1.0


In [14]:
vectors= [
      [1.0, 2.0, 3.0, 4.0],    # batch 0, token 0, Hello!
      [20.0, 9.0, 2.0, 0.1],   # batch 0, token 1, Good 
      [2.0, 4.0, 6.0, 8.0] ,  # batch 0, token 2, Morning.
      [0.0, 5.0, 10.0, 150.0], # batch 1, token 0, Bye!
      [1.0, 1.0, 1.0, 1.0],    # batch 1, token 1, Cya
      [2.0, 3.0, 4.0, 5.0],
      [101,102,103,104]]  # batch 1, token 2, Tomorrow.

In [15]:
for data in vectors:
    normalized_data = standardize_vector(data)
    print("\n")
    print("Original Data:", data)
    print("Mean of original data    :", np.mean(data))
    print("Std Dev of original data :", np.std(data))
    print("Variance Dev of original data :", np.std(data)*np.std(data))
    print("Normalized Data:", normalized_data)
    print("Mean of normalized data (approx 0):", np.mean(normalized_data))
    print("Std Dev of normalized data (approx 1):", np.std(normalized_data))
    print("Variance Dev of normalized data (approx 1):", np.std(normalized_data)*np.std(normalized_data))



Original Data: [1.0, 2.0, 3.0, 4.0]
Mean of original data    : 2.5
Std Dev of original data : 1.118033988749895
Variance Dev of original data : 1.2500000000000002
Normalized Data: [-1.34164079 -0.4472136   0.4472136   1.34164079]
Mean of normalized data (approx 0): 0.0
Std Dev of normalized data (approx 1): 1.0
Variance Dev of normalized data (approx 1): 1.0


Original Data: [20.0, 9.0, 2.0, 0.1]
Mean of original data    : 7.775
Std Dev of original data : 7.797555706758368
Variance Dev of original data : 60.801874999999995
Normalized Data: [ 1.56779899  0.15710051 -0.7406167  -0.9842828 ]
Mean of normalized data (approx 0): -5.551115123125783e-17
Std Dev of normalized data (approx 1): 0.9999999999999999
Variance Dev of normalized data (approx 1): 0.9999999999999998


Original Data: [2.0, 4.0, 6.0, 8.0]
Mean of original data    : 5.0
Std Dev of original data : 2.23606797749979
Variance Dev of original data : 5.000000000000001
Normalized Data: [-1.34164079 -0.4472136   0.4472136   1.34